# 추론 모델 밑바닥부터 5편 실습 노트북 — 6장: 강화학습으로 추론 학습하기(GRPO·RLVR) (한국어 주석판)

**출처:** 원본 노트북 `ch06_main.ipynb`, Sebastian Raschka, https://github.com/rasbt/reasoning-from-scratch , Apache License 2.0. 책 *Build a Reasoning Model (From Scratch)*, Manning.

**변경 사항:** 코난쌤(@conanssam)이 설명 글을 한국어로 옮기고 코드에 한국어 주석을 덧붙였습니다. 코드 동작은 원본과 같습니다.

**쓰는 법**
1. 저장소를 내려받습니다: `git clone https://github.com/rasbt/reasoning-from-scratch.git`
2. 저장소 폴더에서 `uv sync`를 실행해 필요한 패키지를 설치합니다.
3. `uv run jupyter lab`으로 주피터랩을 열고, 이 파일을 `ch06/01_main-chapter-code/` 폴더에 넣어 실행합니다.

**팁:** 학습은 GPU가 있어야 현실적입니다(원작 README도 GPU를 권합니다). 처음에는 스텝 수(`steps`)를 작게 잡아 돌려 보세요.

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a>의 책 <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>에 딸린 실습 코드입니다<br>
<br>코드 저장소: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 6장: 강화학습으로 추론 모델 학습하기

이 노트북에서 쓰는 패키지입니다.

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

# 설치된 패키지 버전을 하나씩 출력합니다
for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F01_raschka.webp" width=600>

&nbsp;
## 6.1 LLM을 위한 강화학습 들어가기

- 추론 시간 확장(inference-time scaling)은 답 하나를 만들 때 계산을 더 써서 추론을 좋게 만듭니다.
- 학습 시간 확장(training-time scaling)은 학습할 때 계산을 더 써서 추론을 좋게 만듭니다. 이 장에서 다루는 것이 바로 이쪽입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F02_raschka.webp" width=600>

- 추론 시간 확장과 학습 시간 확장은 함께 쓸 수 있고, 함께 쓰는 게 좋습니다. 예를 들어 강화학습(RL)으로 추론을 학습시킨 뒤에 추론 시간 기법을 얹을 수 있습니다.
- 실제로 LLM의 강화학습은 사후 학습(post-training) 단계로 씁니다. 사전학습된 모델 위에, 또는 지시 미세조정(instruction fine-tuning) 다음에 붙입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F03_raschka.webp" width=600>

- 사전학습(pre-training)은 다음 토큰 예측으로 폭넓은 지식을 쌓습니다. 반면 강화학습은 답이 맞았는지, 사람이 좋아하는지처럼 문장 전체 단위의 목표를 최적화해서 모델의 행동을 다듬습니다.
- LLM의 강화학습에는 추론 학습과 선호 조정(preference tuning)이 있습니다. 그런데 DeepSeek-R1이 보여 준 것처럼, 추론 중심 강화학습은 사전학습된 기본(base) 모델에 바로 걸 수도 있습니다.
- 기본 모델에 바로 추론을 학습시키면 조금 약하지만 여전히 쓸 만한 모델이 나옵니다. 대신 추론 단계가 무엇을 보태는지 이해하기에는 더 단순한 환경입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F04_raschka.webp" width=600>

&nbsp;
### 6.1.1 원조 강화학습 파이프라인: 사람 피드백 강화학습(RLHF)

- RLHF(reinforcement learning with human feedback)는 2022년 InstructGPT 연구에서 처음 나왔습니다. 사람이 매긴 선호 라벨로 LLM을 학습시킵니다(GPT-3를 처음의 ChatGPT로 바꾼 핵심 단계였습니다).
- 사전학습과 지도 미세조정(supervised fine-tuning)은 다음 토큰 예측을 최적화합니다. 이와 달리 RLHF는 모델 답변에 사람이 매긴 선호 라벨을 기준으로 모델을 최적화합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F05_raschka.webp" width=600>

&nbsp;
### 6.1.2 사람 피드백에서 검증 가능한 보상으로(RLVR)

- RLHF는 보상 모델(reward model)을 따로 학습시켜야 합니다. 이 보상 모델은 크고 비싼 LLM인 경우가 많습니다.
- RLVR(reinforcement learning with verifiable rewards)은 학습된 보상 모델 대신, 자동으로 확인할 수 있고 늘 같은 결과가 나오는 보상을 씁니다. 이것을 검증 가능한 보상(verifiable rewards)이라고 부릅니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F06_raschka.webp" width=600>

- RLVR이 인기를 끈 데는 2025년 DeepSeek-R1의 성공이 컸습니다. 사람 선호 데이터나 학습된 보상 모델 없이도 추론 성능이 높게 나온다는 것을 보여 주었기 때문입니다.
- DeepSeek-R1은 자동으로 검증할 수 있는 보상으로 추론 행동을 학습시켰습니다. 수학 문제는 정답 확인, 프로그래밍 문제는 코드 컴파일이나 실행 결과를 보상으로 썼습니다.
- 이 책은 수학 검증에 집중합니다. 하지만 코드 검증도 기본 생각은 같습니다. 성공/실패라는 두 가지 신호로 보상을 자동으로 계산합니다.

&nbsp;
## 6.2 GRPO로 검증 가능한 보상 강화학습 따라가 보기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F07_raschka.webp" width=600>

- 큰 그림을 보고 강화학습이 LLM 개발 과정의 어디에 들어가는지 알았습니다. 이제 RLVR을 직접 구현해서 추론 모델을 학습시켜 봅니다. DeepSeek-R1-Zero와 비슷하지만 규모는 훨씬 작습니다. 비슷한 규모로 돌리면 GPU 비용만 수십만 달러가 들기 때문입니다.
- LLM의 강화학습은 정책 경사(policy gradient) 알고리즘으로 우리가 학습시킬 LLM을 업데이트합니다. 강화학습에서는 이 LLM을 정책(policy)이라고 부릅니다.
- RLHF에서 많이 쓰는 정책 경사 알고리즘은 PPO(proximal policy optimization)입니다. RLVR에도 같은 알고리즘을 쓸 수 있습니다.
- 그런데 DeepSeek 팀은 DeepSeek-R1 추론 모델을 학습시킬 때 더 단순한 알고리즘을 썼습니다. 바로 GRPO(group relative policy optimization)입니다(DeepSeekMath에서 처음 썼습니다).
- GRPO는 자원을 덜 씁니다. PPO는 가치 함수(value function)를 계산하는 LLM이 하나 더 필요합니다. GRPO는 그게 필요 없습니다. 같은 문제로 뽑은 답 여러 개를 한 그룹 안에서 서로 비교해 학습 신호를 얻기 때문입니다.
- PPO와 GRPO를 나란히 놓고 더 자세히 비교한 내용은 제 글 [The State of Reinforcement Learning for LLM Reasoning](https://magazine.sebastianraschka.com/p/the-state-of-llm-reasoning-model-training)에 있습니다.
- 이 장에서는 GRPO로 RLVR을 구현합니다.
- 다음 장에서는 GRPO를 더 개선해서 학습을 안정시키고 모델 성능을 높이는 방법을 소개합니다.

### 6.2.1 요리사 비유로 보는 GRPO의 큰 그림

- GRPO는 처음 보면 복잡해 보일 수 있습니다. 그래서 이 절은 '요리사와 요리' 비유로 큰 그림부터 봅니다. 용어를 익히고 감을 잡기 위해서입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F08_raschka.webp" width=600>

- LLM 강화학습에서는 대부분 롤아웃(rollout)과 완성(completion)을 같은 뜻으로 씁니다.

### 6.2.2 GRPO의 큰 흐름

- 다음 절들에서 GRPO를 구현하는 기술 로드맵입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F09_raschka.webp" width=600>

&nbsp;
## 6.3 사전학습된 모델 불러오기

- 이 장의 코드는 앞 장들의 코드와 똑같습니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F10_raschka.webp" width=600>

In [ ]:
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()
# 일단 CPU로 고정합니다 (학습 단계에서 다시 get_device()로 바꿉니다)
device = torch.device("cpu")

# 학습의 출발점인 기본(base) 모델과 토크나이저를 불러옵니다
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
# 문제를 모델이 읽는 프롬프트 형식으로 감쌉니다
prompt = render_prompt(raw_prompt)

# 학습 전 기본 모델이 이 문제를 어떻게 푸는지 봅니다
torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

&nbsp;
## 6.4 MATH 학습용 부분 데이터셋 불러오기

- 원래 MATH 데이터셋에서 뽑은 학습용 부분 데이터셋을 씁니다. 앞 장들에서 모델 평가에 쓴 MATH-500 문제는 일부러 뺐기 때문에 겹치지 않습니다(데이터셋을 어떻게 만들었는지는 https://github.com/rasbt/math_full_minus_math500 을 보세요).

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F11_raschka.webp" width=600>

- 아래 `load_math_train` 함수는 3장의 [load_math500_test](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch03.py#L422) 함수와 비슷합니다. 파일 경로만 다릅니다.

In [ ]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    # 이미 받아 둔 파일이 있으면 그것을 읽습니다
    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        # 기본 주소가 안 되면 백업 주소에서 받습니다
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        # 다음에 다시 받지 않도록 내 컴퓨터에 저장해 둡니다
        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

In [ ]:
math_train = load_math_train()

# 학습용 문제가 몇 개인지 확인합니다
print("Dataset size:", len(math_train))

In [ ]:
from pprint import pprint

# 문제 하나가 어떤 필드로 되어 있는지 봅니다
pprint(math_train[4])

- 우리에게 필요한 필드는 `"answer"`와 `"problem"` 두 개뿐입니다.
- `"solution"`(풀이)도 쓰고 싶을 수 있습니다. 하지만 여기서는 모델이 정해진 풀이와 말투를 배우는 대신, 풀이를 자유롭게 탐색하게 두려고 합니다.

&nbsp;
## 6.5 롤아웃 뽑기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F12_raschka.webp" width=600>

- 롤아웃(rollout)은 '모델이 생성한 답'을 가리키는 강화학습 용어입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F13_raschka.webp" width=400>

- 여기서는 `@torch.no_grad`가 필요합니다. 이 부분은 계산 그래프를 만들거나 역전파할 필요가 없기 때문입니다. 그런데 `@torch.inference_mode`는 너무 많은 것을 꺼 버려서 쓸 수 없습니다. 쓰면 다음 오류가 납니다.

> RuntimeError: Inference tensors cannot be saved for backward. Please do not use Tensors created in inference mode in computation tracked by autograd. To work around this, you can make a clone to get a normal tensor and use it in autograd, or use `torch.no_grad()` instead of `torch.inference_mode()`.

In [ ]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


# 롤아웃을 뽑는 동안에는 기울기를 계산하지 않습니다
@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    # KV 캐시로 프롬프트를 한 번에 읽고 마지막 위치의 로짓을 얻습니다
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for step in range(max_new_tokens):
        if temperature and temperature != 1.0:
            # 온도로 확률 분포를 납작하게(또는 뾰족하게) 만듭니다
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        # top-p로 확률이 낮은 꼬리를 잘라 냅니다
        probas = top_p_filter(probas, top_p)
        # 남은 확률대로 다음 토큰 하나를 뽑습니다
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            # 끝(EOS) 토큰이 나오면 생성을 멈춥니다
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
            logits = model(next_token, cache=cache)[:, -1]

    # 프롬프트 + 생성 토큰 전체, 프롬프트 길이, 답 글을 함께 돌려줍니다
    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

- 새로운 내용은 없습니다.
- 위 코드는 앞에서 만들어 온 코드를 더 가볍게 줄인 것입니다. 2장의 [generate_text_basic_stream_cache](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch02.py#L57) 함수에 4장의 온도(temperature)·top-p 샘플링을 바로 합쳤습니다.
- 그리고 토큰을 하나씩 내보내지(yield) 않고 텐서에 모아 둡니다. 생성한 토큰을 실시간으로 출력할 필요가 없기 때문입니다.

In [ ]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

# 롤아웃 하나를 뽑습니다
token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

In [ ]:
# 시드만 바꿔 다른 롤아웃을 뽑아 봅니다
torch.manual_seed(5)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

- 실제로는 sample_response를 여러 번 불러서 롤아웃을 만듭니다.
- 하지만 GRPO 설명을 단순하게 하고 그림 6.13과 맞추려고, 여기서는 모델이 아래 네 답을 만들었다고 가정합니다.

In [ ]:
# 그림 6.13처럼 네 개의 롤아웃이 나왔다고 가정합니다 (정답은 83)
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

&nbsp;
## 6.6 보상 계산하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F14_raschka.webp" width=400>

- 보상은 3장과 비슷하게 단순한 정답 보상입니다.
- 다만 형식 보상도 숨어 있습니다. 최종 답이 `\boxed{}` 형식일 때만 보상 1.0을 줍니다(`fallback=None`으로 처리합니다).

In [ ]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

# 검증 가능한 보상: 맞으면 1.0, 틀리거나 형식이 없으면 0.0
def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None  # Require \boxed{}
    )
    # \boxed{}가 없으면 보상은 0입니다
    if not extracted:
        return 0.0
    # 정답과 같은지 채점합니다
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [ ]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]
rollout_rewards = []

# 롤아웃마다 보상을 계산해 모아 둡니다
for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

- 참고: DeepSeek-R1 팀은 모델을 학습시킬 때 풀이 중간 단계에 점수를 매기는 과정 보상 모델(process reward model)도 써 보았습니다.
- 하지만 잘되지 않았습니다. 그래서 중간 보상 없이 최종 답이 맞았는지만으로 학습시키는 편이 낫다고 결론 내렸습니다.

&nbsp;
## 6.7 이점으로 롤아웃에서 학습 신호 만들기

- GRPO의 "GR"(group relative, 그룹 상대)은 이런 뜻입니다. GRPO는 문제 하나에 답(롤아웃)을 여러 개 만들고, 그 답들을 서로 비교해서 학습 신호를 만듭니다. 이 신호를 이점(advantage)이라고 부릅니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F15_raschka.webp" width=400>

- 식은 아주 간단합니다.

$$\text{advantages}_i = \frac{r_i - \mu_r}{\sigma_r + \epsilon}$$

- 여기서 $r_i$는 $i$번째 롤아웃의 보상, $\mu_r$은 그룹 안 롤아웃들의 평균 보상, $\sigma_r$은 그 표준편차입니다. $\epsilon$은 0으로 나누는 오류를 막으려고 더하는 작은 상수입니다.

In [ ]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

In [ ]:
# 그룹 평균보다 나으면 +, 못하면 - 이점이 됩니다
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

- 한 그룹의 보상이 모두 같으면(예를 들어 전부 0이거나 전부 1이면) 모든 롤아웃 $i$에서 $r_i - \mu_r = 0$이 됩니다.
- 즉, 답이 모두 맞거나 모두 틀리면 모델은 업데이트되지 않습니다.

&nbsp;
## 6.8 문장 전체 로그 확률로 롤아웃 점수 매기기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F16_raschka.webp" width=400>

- 앞 장에서 답 토큰의 토큰별 로그 확률을 계산하는 `avg_logprob_answer` 함수를 만들었습니다.
- 이렇게 평균 낸 로그 확률을 토큰 단위 로그 확률이라고도 부릅니다. LLM의 답에 점수를 매길 때 흔히 씁니다.
- 점수를 매길 때 평균을 쓰는 이유는 길이를 맞춰 주기(length-normalization) 때문입니다.
- 수식으로는 $\frac{1}{T} \sum_{t=1}^{T} \log p_W(y_t \mid y_{<t}, x)$로 씁니다.
- 여기서 $y_1, ..., y_T$는 길이가 $T$인 생성된 답의 토큰입니다. $T, y_{<t}$는 앞서 생성된 모든 토큰, $x$는 입력 프롬프트, $W$는 모델의 가중치 파라미터입니다.
- 수학적으로는 앞 장의 식과 똑같습니다. 생성된 출력 토큰과 입력 프롬프트를 구분하기 쉽게 $x$를 $y$로 바꿨을 뿐입니다.
- 참고용으로 그 함수를 아래에 다시 옮겨 둡니다.

In [ ]:
## Chapter 5

# 5장에서 만든 함수 그대로입니다 (토큰 평균 로그 확률)
@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # Encode prompt and answer tokens separately to get the prompt length later
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # Same as in calc_next_token_logprobas before
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # Index range for positions corresponding to answer tokens
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # Same as before, except for using start and end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Average over the answer token scores
    return torch.mean(next_token_logps).item()

In [ ]:
# 6.5절에서 뽑은 답의 평균 로그 확률을 구합니다
avg_logprob_val = avg_logprob_answer(
                   model, tokenizer, 
                   prompt=prompt,
                   answer=answer_text,
                   device=device) 
print(avg_logprob_val)

- 그런데 GRPO는 위처럼 길이로 나눈 토큰 단위 평균이 아니라, 문장 전체(sequence-level) 로그 확률을 씁니다.
- 토큰 단위 평균은 길이가 다른 답들을 비교할 수 있게 해 주므로 점수 매기기에 좋습니다.
- GRPO에서는 롤아웃 하나가 문장 전체에 대해 보상 하나, 이점 하나를 받습니다. 그래서 기울기(gradient) 크기를 제대로 맞추려면 로그 확률도 문장 전체의 가능도를 나타내야 합니다. 토큰별 로그 확률을 더하면 그 값이 나옵니다.
- 그러지 않고 평균을 내면 학습 신호가 문장 길이에 따라 슬그머니 다시 조정됩니다. 그러면 정책 업데이트가 왜곡되고, 롤아웃이 길수록 더 심해집니다.

- 평균을 빼면 문장 전체 로그 확률로 바꿀 수 있습니다. `torch.mean(next_token_logps)`를 `torch.sum(next_token_logps)`로 바꾸면 됩니다.
- 이미 구한 평균값에 답 토큰 수를 곱해도 평균 내기 전 값이 나옵니다.

In [ ]:
sequence_logprob_val = avg_logprob_val * (len(tokenizer.encode(answer_text)))
print(sequence_logprob_val)

- 이런 문장 전체 로그 확률은 문장 길이 T에 비례해 커집니다.
- 그래서 답이 길수록 로그 확률이 늘 더 큰 음수가 됩니다.
- 그 결과 똑같이 좋은 답 두 개가 있으면 더 짧은(더 싼) 쪽을 고르게 됩니다.
- 로그 확률을 더하면 모델이 더 일찍 멈추도록 이끕니다.

- 앞에서 말했듯 위 함수에서 `torch.mean`을 `torch.sum`으로 바꾸면 됩니다.
- 하지만 앞 장에서는 `@torch.inference_mode()` 데코레이터를 붙여 추론 모드로 돌렸습니다. 이번에는 PyTorch가 기울기를 추적하고 계산해야 하므로 어차피 함수를 다시 정의해야 합니다.
- 또 6.5절의 `sample_response`가 `token_ids`와 `prompt_len`을 돌려주므로, `avg_logprob_answer`에 있던 인코딩과 `full_ids` 계산을 빼서 더 단순하게 만들 수 있습니다.

In [ ]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    # 기울기를 추적하도록 inference_mode 없이 모델을 돌립니다
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # Positions whose next-token probabilities we want
    # These correspond to predicting token_ids[t + 1] from position t
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Sum log-probabilities over the answer tokens
    # 평균(mean) 대신 합(sum)을 씁니다
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

- `torch.sum(next_token_logps)`에 `.item()`을 붙이지 않은 점을 보세요. 그래야 PyTorch가 파이썬 실수(float)가 아니라 텐서를 돌려줍니다. 기울기 계산에 꼭 필요합니다.
- 보다시피 결과값(-16.2998)은 앞에서 `avg_logprob_val`에 답 토큰 수를 곱해 얻은 값(-16.2390)과 거의 같습니다. 작은 차이는 부동소수점 반올림 때문입니다.

- 아래에서는 torch.gather로 함수를 다시 씁니다. PyTorch다운 방식이고 GPU에서 조금 더 잘 최적화됩니다.
- 하지만 두 함수는 수학적으로 같습니다.

In [ ]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    # gather로 각 위치에서 실제 다음 토큰의 로그 확률만 고릅니다
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    # 프롬프트 부분은 빼고 답 토큰만 더합니다
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

In [ ]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

# 네 롤아웃 각각의 문장 전체 로그 확률을 계산합니다
for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

- 짧고 간결한 답일수록 문장 전체 로그 확률이 더 높습니다(덜 음수입니다).
- 가장 낮은 점수는 유일하게 틀린 값(83이 아니라 38)을 담은 답이 받았습니다.
- 정리하면, 로그 확률을 더하면 간결하고 정확한 출력이 유리해집니다.

&nbsp;
## 6.9 GRPO 손실로 이점을 정책 업데이트로 바꾸기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F17_raschka.webp" width=400>

In [ ]:
logps = torch.stack(rollout_logps)
print(logps)

In [ ]:
# 정책 경사 손실: 이점 x 로그 확률의 평균에 음수를 붙입니다
pg_loss = -(advantages.detach() * logps).mean()
print(pg_loss)

- `.detach()`가 필요한 이유는 `advantages`를 고정된 학습 신호로 다루고 싶기 때문입니다. 이렇게 하면 역전파가 로그 확률 쪽으로만 흐릅니다.
- 앞에 음수 부호를 붙이는 이유는 PyTorch 옵티마이저가 기본으로 값을 최소화하기 때문입니다. 여기서는 로그 확률에 이점을 곱한 값을 최대화하고 싶습니다.

- 정책 경사 손실(policy gradient loss)을 수식으로 쓰면 다음과 같습니다.

$$\mathcal{L}_{\mathrm{PG}}
= -\frac{1}{N} \sum_{i=1}^{N} A_i \sum_{t=1}^{T_i} \log p_W\!\left( y_t^{(i)} \mid y_{<t}^{(i)}, x^{(i)} \right)$$

- $N$은 배치 안의 롤아웃 수입니다.
- $y_1^{(i)}, ..., y_{T_i}^{(i)}$는 길이가 $T_i$인 $i$번째 생성 답의 토큰입니다.
- $y_{<t}^{(i)}$는 그 답에서 앞서 생성된 모든 토큰입니다.
- $x^{(i)}$는 $i$번째 롤아웃의 입력 프롬프트입니다.
- $p_W$는 모델의 정책입니다. 즉 가중치 $W$로 정해지는 다음 토큰 확률 분포입니다.
- $A_i$는 $i$번째 롤아웃 전체에 매긴 이점입니다.
- 안쪽 합은 롤아웃 하나의 문장 전체 로그 확률을 계산합니다.
- 바깥쪽 평균은 이점을 곱한 로그 확률을 롤아웃 전체에 걸쳐 평균 냅니다.

&nbsp;
## 6.10 GRPO 한 스텝으로 모두 합치기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F18_raschka.webp" width=400>

In [ ]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    # 그룹 안에서 비교해야 하므로 롤아웃은 2개 이상이어야 합니다
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    # 롤아웃을 뽑는 동안은 평가 모드로 둡니다
    model.eval()

    for _ in range(num_rollouts):
        # Stage 1: generate rollouts
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Stage 2: compute rewards
        reward = reward_rlvr(text, example["answer"])
        
        # Stage 4: compute logprobs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        # 나중에 출력해 볼 수 있게 답·보상·길이를 기록합니다
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        # 원래 학습 모드였다면 되돌립니다
        model.train()

    # Stage 2: collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # Stage 3: compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Stage 4: collect all logprobs
    logps = torch.stack(roll_logps)

    # Stage 5: compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    # loss_tensor는 역전파에 쓰고, 나머지는 기록용입니다
    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

- 코드 주석의 단계(Stage)는 GRPO 그림의 단계와 짝을 이룹니다.
- 코드 주석에서는 1단계 다음에 (3, 4단계가 아니라) 2, 4단계가 옵니다. 이렇게 해야 코드가 더 단순해집니다(for 반복문을 여러 개 만들지 않아도 됩니다).

In [ ]:
torch.manual_seed(123)

# 문제 하나로 GRPO 손실을 한 번 계산해 봅니다
stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

&nbsp;
## 6.11 GRPO 학습 루프 구현하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F19_raschka.webp" width=600>

- 이미 자원이 많이 들기 때문에 배치(batching)는 건너뜁니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F20_raschka.webp" width=400>

In [ ]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    # 스텝 수를 안 주면 데이터셋을 한 바퀴 돕니다
    if steps is None:
        steps = len(math_data)

    # Stage 1: initialize optimizer
    # (the model was already initialized outside the function)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # Stage 2: Iterate over training steps
        for step in range(steps):

            # Stage 3: Reset loss gradient
            # (it's best practice to do this at the beginning of each step)
            optimizer.zero_grad()

            current_step = step + 1
            # 스텝마다 문제를 하나씩 차례로 꺼냅니다
            example = math_data[step % len(math_data)]

            # Stage 4: calculate GRPO loss
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # Stage 5: Backward pass to calculate loss gradients
            stats["loss_tensor"].backward()

            # Clip large gradients to improve training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Stage 6: Update model weights using loss gradients
            optimizer.step()

            # Stage 7: Collect rewards, response lengths, and losses
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) 
                if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps, stats["loss"],
                reward_avg, avg_response_len,
            )

            # Print step metrics
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # Sample outputs (every 10 steps) to check if model
            # generates coherent text
            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            # Stage 8: Save model checkpoint
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    # Save a model checkpoint if we interrupt the training early
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model


# 모델 가중치를 .pth 파일로 저장합니다
def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


# 스텝별 손실·보상·답 길이를 CSV에 한 줄씩 덧붙입니다
def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

- 4단계인 GRPO 손실 계산을 빼면, 나머지는 모두 심층 신경망(LLM 포함)을 학습시킬 때 쓰는 표준 학습 루프입니다.
- `append_csv_metrics`는 기록을 남기려고 결과를 CSV 파일에 저장합니다(7장에서 결과를 그래프로 그릴 때도 씁니다).
- PyTorch로 신경망을 학습시키는 기본은 제 글 [PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/)의 3~8절을 보세요.

- PyTorch 2.7 대신 2.13으로 코드를 돌리면 5번째 스텝부터 숫자가 조금씩 달라집니다.

In [ ]:
# 학습은 GPU(있으면)로 돌립니다
device = get_device()
model.to(device)

torch.manual_seed(0)

# 50스텝 학습 (처음에는 steps를 작게 줄여 돌려 보세요)
train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

- 위 코드를 돌리다 메모리 문제가 생기면 롤아웃 수(예: `num_rollouts=2`)와 롤아웃당 토큰 수(예: `max_new_tokens=128`)를 줄여 보세요.
- 다만 꽤 괜찮은 모델을 얻으려면 적어도 `num_rollouts=8`과 `max_new_tokens=512`가 필요합니다.
- 가진 하드웨어로 돌릴 수 없어도 걱정하지 마세요. 다음 절에서 미리 학습된 체크포인트를 내려받는 방법을 보여 줍니다.

- 어느 쪽이든 코드는 아마 아주 느리게 돌 것입니다. GRPO는 자원을 많이 쓰는 방법이기 때문입니다.
- 언제든 실행을 멈출 수 있습니다. 멈추면 가장 최근 모델 체크포인트를 `checkpoints` 폴더에 저장합니다.
- 클라우드 GPU를 쓰고 싶다면 [GPU Cloud Resources](../../ch02/02_setup-tips/gpu-instructions.md) 문서의 추천을 보세요.

- 이 코드는 배치 학습을 지원하지 않습니다.
- 코드를 더 단순하고 읽기 쉽게 하려고 일부러 그렇게 했습니다. 또 (길 수도 있는) 롤아웃 여러 개를 뽑는 것만으로도 이미 자원이 많이 들기 때문입니다.
- GPU가 여러 개 있다면, 배치와 멀티 GPU를 지원하는 선택 버전 코드를 부록 자료 [../02_rlvr_grpo_scripts_intro](../02_rlvr_grpo_scripts_intro)에서 찾을 수 있습니다. 모델을 더 빨리 학습시킵니다.

&nbsp;
## 6.12 저장한 모델 체크포인트 불러와 평가하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F21_raschka.webp" width=600>

- 저장한 체크포인트는 2장에서 설명한 model.load_state_dict(torch.load(model_path))로 불러올 수 있습니다. 여기서 model_path는 체크포인트 ".pth" 파일 경로입니다.
- 이 체크포인트 파일은 3장의 모델 평가 도구와도 호환됩니다.
- 편하게 쓰려면 3장 부록 자료에 있는 평가 스크립트를 쓰면 됩니다.

```python
uv run ../../ch03/02_math500-verifier-scripts/evaluate_math500.py \
--dataset_size 500 \
--which_model base \
--checkpoint_path checkpoints/qwen3-0.6B-rlvr-grpo-step00050.pth
```


- 시간이 너무 오래 걸려서 내 컴퓨터로 GRPO 학습을 돌리고 싶지 않다면, 제가 올려 둔 체크포인트를 [rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl)에서 내려받아도 됩니다(원하는 체크포인트 파일을 누른 뒤 [download](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/resolve/main/grpo_original_no_kl/qwen3-0.6B-rlvr-grpo-step00050.pth?download=true) 버튼을 누르세요).
- 편하게 쓰도록 아래 파이썬 코드로 바로 내려받을 수도 있습니다.

In [ ]:
from reasoning_from_scratch.qwen3 import download_qwen3_grpo_checkpoints

# KL 없는 GRPO로 50스텝 학습한 체크포인트를 내려받습니다
download_qwen3_grpo_checkpoints(grpo_type="no_kl", step="00050")

|      | 방법                                   | 스텝 | 최대 토큰 | 롤아웃 수 | MATH-500 정확도 | 평균 토큰 수 |
| ---- | -------------------------------------- | ---- | ---------- | ------------ | ------------ | --------------- |
| 1    | 기본(base) 모델 (3장)                  | -    |            |              | 15.2%        | 78.85           |
| 2    | 추론(reasoning) 모델 (3장)             | -    |            |              | 48.2%        | 1369.79         |
| 3    | GRPO 원본, KL 없음 (이 장)             | 50   | 512        | 8            | 47.4%        | 586.11          | 

- 위 표를 보면, 기본 모델(1행)에서 출발해 50스텝만 학습시킨 모델(3행)이 원래 추론 모델(2행)과 거의 비슷하게 잘합니다.
- 더 오래 학습시킨다고 모델이 좋아지지는 않을 수 있고, 오히려 나빠질 수도 있습니다. GRPO는 비교적 불안정하기 때문입니다. 다음 장에서 GRPO 알고리즘을 개선하는 요령을 더 소개합니다.

&nbsp;
## 6.13 정리

- 이 절에는 코드가 없습니다.